<a href="https://colab.research.google.com/github/hwanginseo04/-/blob/main/%EC%95%B1%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D_%EC%A4%91%EA%B0%84%EA%B3%A0%EC%82%AC-%EC%B5%9C%EC%A2%85-.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. 필수 라이브러리 설치
!pip install fastapi uvicorn httpx beautifulsoup4 gradio pandas pyngrok nest_asyncio

In [3]:
# 2. 필수 라이브러리 설치
!pip install fastapi uvicorn httpx beautifulsoup4 gradio pandas pyngrok nest_asyncio deep-translator

In [4]:
import sqlite3, httpx, asyncio, nest_asyncio, uvicorn, os, re, subprocess
import pandas as pd
import gradio as gr
from bs4 import BeautifulSoup
from fastapi import FastAPI, HTTPException, Path
from pydantic import BaseModel, Field
from gradio import mount_gradio_app
from collections import Counter
from pyngrok import ngrok
from deep_translator import GoogleTranslator

# --- [설정] ---
nest_asyncio.apply()
app = FastAPI(title="📌 명언 관리 시스템", description="중간고사 과제 최종본")
DB_NAME = "final_assignment_v2.db"

# --- [DB 및 수집: 기존 로직 100% 유지] ---
def scrape():
    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("DROP TABLE IF EXISTS quotes")
        conn.execute("CREATE TABLE quotes (id INTEGER PRIMARY KEY AUTOINCREMENT, text TEXT, translation TEXT, author TEXT, tags TEXT)")

    data = []; page = 1; translator = GoogleTranslator(source='en', target='ko')
    print("⏳ 영어 명언을 한글로 번역하며 20개 채우는 중...")
    with httpx.Client() as client:
        while len(data) < 20:
            res = client.get(f"https://quotes.toscrape.com/page/{page}/")
            soup = BeautifulSoup(res.text, 'html.parser')
            items = soup.select(".quote")
            if not items: break
            for item in items:
                if len(data) >= 20: break
                t = item.find(class_="text").get_text(strip=True)
                a = item.find(class_="author").get_text(strip=True)
                try: trans = translator.translate(t)
                except: trans = "번역 오류"
                data.append((t, trans, a, "명언"))
            page += 1
    with sqlite3.connect(DB_NAME) as conn:
        conn.executemany("INSERT INTO quotes (text, translation, author, tags) VALUES (?, ?, ?, ?)", data)
    print("✅ 20개 수집 완료!")

# --- [API 명세서용 CRUD: 기존 로직 100% 유지] ---
class QuoteModel(BaseModel):
    text: str = Field(..., description="영어 원문")
    translation: str = Field(..., description="한글 번역")
    author: str = Field(..., description="작가")
    tags: str = Field(..., description="태그")

@app.get("/quotes", tags=["명언 관리"], summary="목록 조회")
def list_q(): return pd.read_sql_query("SELECT * FROM quotes", sqlite3.connect(DB_NAME)).to_dict(orient="records")

@app.post("/quotes", tags=["명언 관리"], summary="추가")
def add_q(q: QuoteModel):
    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("INSERT INTO quotes (text, translation, author, tags) VALUES (?,?,?,?)", (q.text, q.translation, q.author, q.tags))
    return {"msg": "추가됨"}

@app.put("/quotes/{qid}", tags=["명언 관리"], summary="수정")
def update_q(qid: int, q: QuoteModel):
    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("UPDATE quotes SET text=?, translation=?, author=?, tags=? WHERE id=?", (q.text, q.translation, q.author, q.tags, qid))
    return {"msg": "수정됨"}

@app.delete("/quotes/{qid}", tags=["명언 관리"], summary="삭제")
def delete_q(qid: int):
    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("DELETE FROM quotes WHERE id=?", (qid,))
    return {"msg": "삭제됨"}

# --- [🎨 시각화 UI: 사진처럼 나오도록 추가] ---
def get_ui_data(search_tag=""):
    with sqlite3.connect(DB_NAME) as conn:
        query = "SELECT id as 'ID', text as '원문', translation as '번역', author as '작가', tags as '태그' FROM quotes"
        if search_tag:
            query += f" WHERE tags LIKE '%{search_tag}%'"
        return pd.read_sql_query(query, conn)

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎓 기초고사특징: FastAPI 기반 명언 관리 시스템")
    gr.Markdown("시각화 UI 주소는 **/ui** 입니다. API 명세서는 **/docs** 입니다.")

    with gr.Row():
        tag_input = gr.Textbox(label="🔍 카테고리(태그) 검색", placeholder="예: 명언")
        search_btn = gr.Button("데이터 조사 및 조회 🚀", variant="primary")

    output_df = gr.Dataframe(value=get_ui_data(), interactive=False)

    # 버튼 클릭 시 데이터 갱신 (이게 없으면 데이터가 안 뜹니다)
    search_btn.click(fn=get_ui_data, inputs=tag_input, outputs=output_df)

# API 명세서와 UI를 동시에 사용하기 위해 마운트
app = mount_gradio_app(app, demo, path="/ui")

# --- [실행] ---
if __name__ == "__main__":
    try:
        ngrok.kill()
        subprocess.run(["pkill", "-9", "ngrok"], check=False)
    except: pass

    scrape()
    ngrok.set_auth_token("3Ct6SFB6shg3RDWSMhpGueCOgkP_7J3tbKU9GwPvar5pGQpwh")

    # ngrok 연결 (실행할 때마다 주소가 새로 바뀝니다!)
    url = ngrok.connect(8000).public_url

    print("\n" + "🚀" * 30)
    print(f"📖 API 명세서 (Swagger): {url}/docs")
    print(f"🎨 시각화 UI (Gradio): {url}/ui")
    print("🚀" * 30 + "\n")
    print("⚠️ ERR_NGROK_3200 방지: 반드시 위 파란색 새 주소로 접속하세요!")

    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)
    loop = asyncio.get_event_loop()
    loop.create_task(server.serve())

/tmp/ipykernel_33858/1917515988.py:79: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


new /ui
⏳ 영어 명언을 한글로 번역하며 20개 채우는 중...
✅ 20개 수집 완료!

🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
📖 API 명세서 (Swagger): https://diagnoses-backboard-respect.ngrok-free.dev/docs
🎨 시각화 UI (Gradio): https://diagnoses-backboard-respect.ngrok-free.dev/ui
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀

⚠️ ERR_NGROK_3200 방지: 반드시 위 파란색 새 주소로 접속하세요!
